In [ ]:
import warnings
import healpy as hp
import bokeh
import pandas as pd
import panel as pn
from astropy.time import Time
import astropy.coordinates


from uranography.api import (
    Planisphere,
    ArmillarySphere,
    MollweideMap,
    split_healpix_by_resolution,
    CameraFootprintPerimeter,
)

In [ ]:
pn.extension()

In [ ]:
night = Time("2026-05-30", location=astropy.coordinates.EarthLocation.of_site("Cerro Pachon"))

In [ ]:
warnings.filterwarnings(
    'ignore',
    message=r'.*Tried to get polar motions for times after IERS data is valid.*')

# Survey footprint

Get a survey footprint (in heapix format):

In [ ]:
nside = 64
footprint = hp.ud_grade(hp.read_map('sample_survey_footprint.fits'), nside)

In [ ]:
low_nside = 16
this_footprint = footprint.copy()
this_footprint[this_footprint == 0] = hp.UNSEEN
footprint_high, footprint_low = split_healpix_by_resolution(
    this_footprint, low_nside, nside
)

# Plotting visits for a night, with bells and whistles

Start with a text file with the list of exposures we want to plot:

In [ ]:
!head exposures.txt

Read it, and add the camera perimeter:

In [ ]:
visits = pd.read_csv('exposures.txt', delim_whitespace=True, header=0, index_col=False)

camera_perimeter = CameraFootprintPerimeter('rubin_idealized.txt')
visits["ra"], visits["decl"] = camera_perimeter(
    visits.boresight_ra, visits.boresight_decl, visits.rotation
)
visits

In [ ]:
initial_mjd = visits.mjd.max()

In [ ]:
# Use the smallest categorical bokeh palette for the number of bands
# actually used
used_bands = [b for b in "ugrizy" if b in visits['band'].values]
num_bands = len(used_bands)
base_palette = bokeh.palettes.Spectral
try:
    band_palette = dict(zip(used_bands, base_palette[num_bands]))
except KeyError:
    band_palette = dict(zip(used_bands, base_palette[3][:num_bands]))

visits["color"] = visits.band.map(band_palette)

Sun and moon coordinates:

In [ ]:
sun_coords = astropy.coordinates.get_body('sun', night)
moon_coords = astropy.coordinates.get_body('moon', night)

Include the survey footprint:

In [ ]:
nside = 32
footprint = hp.ud_grade(hp.read_map('sample_survey_footprint.fits'), nside)

low_nside = 16
this_footprint = footprint.copy()
this_footprint[this_footprint == 0] = hp.UNSEEN
footprint_high, footprint_low = split_healpix_by_resolution(
    this_footprint, low_nside, nside
)

In [ ]:
psphere_pix_width = 256

In [ ]:
data_source = {}

asphere_plot = bokeh.plotting.figure(
    frame_width=psphere_pix_width,
    frame_height=psphere_pix_width,
    match_aspect=True,
    title="Armillary Sphere",
)

asphere = ArmillarySphere(mjd=initial_mjd, plot=asphere_plot)

Add the footprint in the background:

In [ ]:
# Faint gray footprint in the background
cmap = bokeh.transform.linear_cmap(
    "value", list(reversed(bokeh.palettes.Greys256))[:32], 0, 1
)
data_source['footprint_high'], cm, gl = asphere.add_healpix(footprint_high, nside=nside, cmap=cmap)
data_source['footprint_low'], cm, gl = asphere.add_healpix(footprint_low, nside=low_nside, cmap=cmap)

Create data sources for each band:

In [ ]:
visit_ds = {}
for band in band_palette.keys():
    band_visits = visits.query(f"band=='{band}'")
    if len(band_visits)>0:
        visit_ds[band] = bokeh.models.ColumnDataSource(visits.query(f"band=='{band}'"))
        asphere.connect_controls(visit_ds[band])

Create a transform to hide visits not made until after the date on the MJD slider:

In [ ]:
past_future_js = """
const result = new Array(xs.length)
for (let i = 0; i < xs.length; i++) {
  if (mjd_slider.value >= xs[i]) {
    result[i] = past_value
  } else {
    result[i] = future_value
  }
}
return result
"""

past_future_transform = bokeh.models.CustomJSTransform(
            args=dict(
                mjd_slider=asphere.controls['mjd'],
                past_value=0.5,
                future_value=0.0
            ),
            v_func=past_future_js,
        )

In [ ]:
recent_js = """
const result = new Array(xs.length)
for (let i = 0; i < xs.length; i++) {
  if (mjd_slider.value < xs[i]) {
    result[i] = 0
  } else {
    result[i] = Math.max(0, max_value * (1 - (mjd_slider.value - xs[i]) / scale))
  }
}
return result
"""

recent_transform = bokeh.models.CustomJSTransform(
            args=dict(
                mjd_slider=asphere.sliders['mjd'],
                max_value=1.0,
                scale=2.0/(24*60)
            ),
            v_func=recent_js,
        )

Actually add the visits:

In [ ]:
# The visits
for band, band_visit_ds in visit_ds.items():
    asphere.plot.patches(
        xs=asphere.proj_transform('x', band_visit_ds),
        ys=asphere.proj_transform('y', band_visit_ds),
        source=band_visit_ds,
        fill_alpha=bokeh.transform.transform('mjd', past_future_transform),
        line_alpha=bokeh.transform.transform('mjd', recent_transform),
        line_color="black",
        line_width=2,
        fill_color="color",
        legend_label=band
    )

Add standard decorations and guides:

In [ ]:
# The sun
data_source['sun'] = asphere.add_marker(
    sun_coords.ra.deg,
    sun_coords.dec.deg,
    name="Sun",
    glyph_size=15,
    circle_kwargs={"color": "yellow", "legend_label": "Sun"},
)

# The moon
data_source['moon'] = asphere.add_marker(
    moon_coords.ra.deg,
    moon_coords.dec.deg,
    name="Moon",
    glyph_size=15,
    circle_kwargs={"color": "orange", "legend_label": "Moon"},
)

asphere.add_graticules()
asphere.add_ecliptic(color="lightgreen", legend_label="Ecliptic")
asphere.add_galactic_plane(color="lightblue", legend_label="Galactic plane")
data_source['horizon'] = asphere.add_horizon(line_kwargs={"legend_label": "Horizon"})
data_source['zd70'] = asphere.add_horizon(
    zd=70,
    line_kwargs={"color": "red", "line_width": 2, "legend_label": "ZD=70" + u'\N{DEGREE SIGN}'}
)

# Tweak the MJD slider end points to match the first and last visit
asphere.controls['mjd'].start = visits.mjd.min()
asphere.controls['mjd'].end = visits.mjd.max()
asphere.controls['mjd'].value = initial_mjd
asphere.controls['mjd'].step = 40./(24*60*60)

# Add a datetime slider
asphere.add_datetime_slider()

# Add equatorial sliders
asphere.add_eq_sliders()

asphere.plot.add_layout(asphere.plot.legend[0], "right")
# pn.Row(asphere.figure)

Show the planisphere beside the armillary sphere:

In [ ]:
psphere_plot = bokeh.plotting.figure(
    frame_width=psphere_pix_width,
    frame_height=psphere_pix_width,
    match_aspect=True,
    title="Armillary Sphere",
)

psphere = Planisphere(mjd=initial_mjd, plot=psphere_plot)
psphere.controls['mjd'] = asphere.controls['mjd']
psphere.add_healpix(data_source['footprint_high'], cmap=cmap)
psphere.add_healpix(data_source['footprint_low'], cmap=cmap)

# The visits
for band, band_visit_ds in visit_ds.items():
    psphere.plot.patches(
        xs=psphere.proj_transform('x', band_visit_ds),
        ys=psphere.proj_transform('y', band_visit_ds),
        source=band_visit_ds,
        fill_alpha=bokeh.transform.transform('mjd', past_future_transform),
        line_alpha=bokeh.transform.transform('mjd', recent_transform),
        line_color="black",
        line_width=2,
        fill_color="color",
    )

psphere.add_marker(
    data_source=data_source['sun'],
    name="Sun",
    glyph_size=15,
    circle_kwargs={"color": "yellow"},
)

psphere.add_marker(
    data_source=data_source['moon'],
    name="Moon",
    glyph_size=15,
    circle_kwargs={"color": "orange"},
)

psphere.add_graticules()
psphere.add_ecliptic(color="lightgreen")
psphere.add_galactic_plane(color="lightblue")
psphere.add_horizon(data_source=data_source["horizon"])
psphere.add_horizon(
    data_source=data_source["zd70"], line_kwargs={"color": "red", "line_width": 2}
)

In [ ]:
msphere_plot = bokeh.plotting.figure(
    frame_width=psphere_pix_width * 2,
    frame_height=psphere_pix_width,
    match_aspect=False,
    title="Mollweide",
)

msphere = MollweideMap(mjd=initial_mjd, plot=msphere_plot)
msphere.controls['mjd'] = asphere.controls['mjd']
msphere.add_healpix(data_source['footprint_high'], cmap=cmap)
msphere.add_healpix(data_source['footprint_low'], cmap=cmap)

# The visits
for band, band_visit_ds in visit_ds.items():
    msphere.plot.patches(
        xs=msphere.proj_transform('x', band_visit_ds),
        ys=msphere.proj_transform('y', band_visit_ds),
        source=band_visit_ds,
        fill_alpha=bokeh.transform.transform('mjd', past_future_transform),
        line_alpha=bokeh.transform.transform('mjd', recent_transform),
        line_color="black",
        line_width=2,
        fill_color="color",
    )

msphere.add_marker(
    data_source=data_source['sun'],
    name="Sun",
    glyph_size=15,
    circle_kwargs={"color": "yellow"},
)

msphere.add_marker(
    data_source=data_source['moon'],
    name="Moon",
    glyph_size=15,
    circle_kwargs={"color": "orange"},
)

msphere.add_graticules()
msphere.add_ecliptic(color="lightgreen")
msphere.add_galactic_plane(color="lightblue")
msphere.add_horizon(data_source=data_source["horizon"])
msphere.add_horizon(
    data_source=data_source["zd70"], line_kwargs={"color": "red", "line_width": 2}
)

In [ ]:
viewable = pn.Column(
    msphere.figure,
    pn.Row(psphere.figure, asphere.figure)
)

display(viewable)
asphere.update(viewable)
psphere.update(viewable)
pn.io.push_notebook(viewable)

In [ ]:
fname = "fancy_example.html"

In [ ]:
viewable.save(fname, embed=True)

In [ ]:
with open(fname, 'r', encoding="utf-8") as in_file:
    html_text = in_file.read()

updated_html = html_text.replace(
    "<script",
    """<script src="https://unpkg.com/gpu.js@latest/dist/gpu-browser.min.js"></script>
    <script""",
    1)

with open(fname, 'w', encoding='utf-8') as out_file:
    out_file.write(updated_html)

In [ ]:
print(msphere.update_js_fnames)

In [ ]:
%pinfo msphere